# Notebook 10 — Real Data: All Methods Compared

This notebook runs the full QNLEP method suite on **real and non-trivial datasets**,
replacing the sin(x) baseline used in earlier notebooks.

**Why real data matters for QNLEP:**
The sin(kx) suite in notebook 03 measures theoretical Fourier capacity (NLC),
which is the right tool for that metric. But for TER and NSS, performance on real
data is what validates the protocol's practical relevance.

**Methods:**
| Method | Non-linearity source | New? |
|---|---|---|
| Data Re-uploading | Fourier encoding | existing |
| Ancilla Tracing | CPTP partial trace | existing |
| Strongly Entangling Layers | Entangling depth | **new** |
| QCNN | Convolutional structure | **new** |
| Quantum Reservoir (QRC) | Fixed random dynamics | **new** |
| Classical MLP | ReLU (reference floor) | reference |

**Datasets:**
| Dataset | Type | Features | Difficulty |
|---|---|---|---|
| `make_circles` | Binary classification | 2 | High (non-linear boundary) |
| `make_moons` | Binary classification | 2 | Medium |
| Iris binary (1 vs 2) | Binary classification | 4 | Medium |
| Mackey-Glass | Regression/time-series | 1 | High (chaotic) |
| x·sin(x) | Regression | 1 | Medium (amplitude growth) |

**Metrics per task:** accuracy (classification), R² (regression), convergence steps.

**Regression targets:** `y` is scaled to **[-1, 1]** using train-set min–max so
MSE training matches a typical Pauli-Z readout scale (inputs `X` stay in **[-π, π]** for encoding).

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, r2_score
import json, pathlib, os

from src.qnlep_run_paths import protocol_results_dir, protocol_n_qubits, protocol_n_layers

RESULTS_DIR = protocol_results_dir()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
pathlib.Path('../figures').mkdir(parents=True, exist_ok=True)

RUN_SEED = int(os.environ.get("QNLEP_SEED", "42"))

# Circuits
from src.circuits.data_reuploading  import build_circuit as build_reuploading
from src.circuits.ancilla_tracing   import build_circuit as build_ancilla
from src.circuits.ancilla_reuploading import build_circuit as build_ancilla_reupload
from src.circuits.strongly_entangling import build_circuit as build_sel, param_shape as sel_shape
from src.circuits.qcnn              import build_circuit as build_qcnn,  param_shape as qcnn_shape
from src.circuits.quantum_reservoir import build_circuit as build_qrc,   param_shape as qrc_shape

# Datasets
from src.datasets import (
    load_circles, load_moons, load_iris_binary,
    load_mackey_glass, load_regression_1d,
)

np.random.seed(RUN_SEED)
torch.manual_seed(RUN_SEED)


def qnlep_env_flag(name: str) -> bool:
    v = os.environ.get(name)
    return (v or "").strip().lower() in ("1", "true", "yes", "on")


## 1. Method registry

Each entry: `(builder_fn, param_shape_fn or None, label)`.
For circuits using the standard `(n_layers, n_qubits, 2)` shape,
`param_shape_fn` is `None` and we use the default.

In [ ]:
N_QUBITS = protocol_n_qubits(4)
N_LAYERS = protocol_n_layers(3)
N_STEPS = int(os.environ.get("QNLEP_REAL_DATA_N_STEPS", "72"))
LR       = 0.05

def default_shape(n_qubits, n_layers): return (n_layers, n_qubits, 2)

METHOD_REGISTRY = {
    'data_reuploading':       (build_reuploading, default_shape),
    'ancilla_tracing':        (build_ancilla,     default_shape),
    'ancilla_reuploading':     (build_ancilla_reupload, default_shape),
    'strongly_entangling':    (build_sel,         sel_shape),
    'qcnn':                   (build_qcnn,        qcnn_shape),
    'quantum_reservoir':      (build_qrc,         qrc_shape),
}

if qnlep_env_flag("QNLEP_REAL_DATA_FAST"):
    if "QNLEP_REAL_DATA_N_STEPS" not in os.environ:
        N_STEPS = min(N_STEPS, 22)
    _slow = ("quantum_reservoir", "qcnn", "strongly_entangling")
    METHOD_REGISTRY = {k: v for k, v in METHOD_REGISTRY.items() if k not in _slow}

_ms = os.environ.get("QNLEP_REAL_DATA_METHODS", "").strip()
if _ms:
    _allow_m = {x.strip() for x in _ms.split(",") if x.strip()}
    METHOD_REGISTRY = {k: v for k, v in METHOD_REGISTRY.items() if k in _allow_m}

def build_all_circuits(n_qubits=N_QUBITS, n_layers=N_LAYERS):
    """Instantiate all circuits and return (circuit, param_shape) pairs."""
    built = {}
    for name, (builder, shape_fn) in METHOD_REGISTRY.items():
        circuit = builder(n_qubits, n_layers)
        shape   = shape_fn(n_qubits, n_layers)
        built[name] = (circuit, shape)
        print(f"  {name:25s}  param shape={shape}  n_params={int(np.prod(shape))}")
    return built

print("Building circuits:")
CIRCUITS = build_all_circuits()

## 2. Shared training functions

In [ ]:
def bce_loss(preds, y):
    """Binary cross-entropy for classification. Preds ∈ [-1,1] mapped to [0,1] via sigmoid."""
    probs = (pnp.tanh(preds) + 1) / 2
    probs = pnp.clip(probs, 1e-6, 1 - 1e-6)
    return -pnp.mean(y * pnp.log(probs) + (1 - y) * pnp.log(1 - probs))


def train_circuit(circuit, param_shape, X_train, y_train,
                  task='classification', n_steps=N_STEPS, lr=LR):
    """
    Train a circuit on (X_train, y_train).
    task: 'classification' uses BCE; 'regression' uses MSE.
    X_train: (n, n_features) — each row is a data point.
    Returns (weights, loss_history, steps_to_90pct).
    """
    weights = pnp.array(np.random.normal(0, 0.1, param_shape), requires_grad=True)
    loss_fn = bce_loss if task == 'classification' else \
              (lambda p, y: pnp.mean((p - y) ** 2))

    n_features = X_train.shape[1]

    def cost(w):
        # Encode each sample: use first feature for 1-feature circuits,
        # or pass features[0] (circuits use scalar x — multi-feature handled below)
        preds = pnp.array([circuit(w, float(X_train[i, 0])) for i in range(len(X_train))])
        return loss_fn(preds, y_train)

    opt     = qml.AdamOptimizer(stepsize=lr)
    history = []
    steps_to_thresh = n_steps
    best_acc = 0.0

    for step in range(n_steps):
        weights, loss = opt.step_and_cost(cost, weights)
        history.append(float(loss))

        # Track threshold crossing for TER
        if steps_to_thresh == n_steps and step > 5:
            preds = np.array([float(circuit(weights, float(X_train[i, 0])))
                              for i in range(len(X_train))])
            if task == 'classification':
                acc = accuracy_score(y_train, (preds > 0).astype(int))
                if acc >= 0.90:
                    steps_to_thresh = step + 1
            else:
                r2 = r2_score(y_train, preds)
                if r2 >= 0.80:
                    steps_to_thresh = step + 1

    return np.array(weights), history, steps_to_thresh


def evaluate_circuit(circuit, weights, X, y, task='classification'):
    preds = np.array([float(circuit(weights, float(X[i, 0]))) for i in range(len(X))])
    if task == 'classification':
        return accuracy_score(y, (preds > 0).astype(int))
    else:
        return r2_score(y, preds)

## 3. Classical MLP reference

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features=1, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),      nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    def forward(self, x): return self.net(x)


def train_mlp(X_train, y_train, task='classification',
              hidden=16, n_steps=N_STEPS, lr=LR):
    model = MLP(in_features=X_train.shape[1], hidden=hidden)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    Xtr   = torch.tensor(X_train, dtype=torch.float32)
    ytr   = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    loss_fn = nn.BCEWithLogitsLoss() if task == 'classification' else nn.MSELoss()
    for _ in range(n_steps):
        opt.zero_grad()
        loss = loss_fn(model(Xtr), ytr)
        loss.backward()
        opt.step()
    return model


def eval_mlp(model, X, y, task='classification'):
    with torch.no_grad():
        preds = model(torch.tensor(X, dtype=torch.float32)).numpy().ravel()
    if task == 'classification':
        return accuracy_score(y, (preds > 0).astype(int))
    else:
        return r2_score(y, preds)

In [ ]:
# Hybrid QCNN: data re-uploading VQC as feature map + small classical ReLU head
# (notebook 06 pattern). Uses the same scalar encoding as PQCs: first input feature only.

CLASSICAL_DIM = 16

_dev_h = qml.device("default.qubit", wires=N_QUBITS)


@qml.qnode(_dev_h, interface="torch", diff_method="backprop")
def hybrid_quantum_features(x, q_weights):
    for l in range(N_LAYERS):
        for w in range(N_QUBITS):
            qml.RY(x, wires=w)
            qml.RY(q_weights[l, w, 0], wires=w)
            qml.RZ(q_weights[l, w, 1], wires=w)
        for w in range(N_QUBITS - 1):
            qml.CNOT(wires=[w, w + 1])
    return [qml.expval(qml.PauliZ(w)) for w in range(N_QUBITS)]


class HybridHQNN(nn.Module):
    """Quantum features + classical head; input is (batch, 1) — first feature column."""

    def __init__(self, hidden: int = CLASSICAL_DIM):
        super().__init__()
        self.q_weights = nn.Parameter(
            torch.tensor(
                np.random.normal(0, 0.1, (N_LAYERS, N_QUBITS, 2)),
                dtype=torch.float32,
            )
        )
        self.head = nn.Sequential(
            nn.Linear(N_QUBITS, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x_batch):
        feats = torch.stack(
            [
                torch.stack(hybrid_quantum_features(x_batch[i, 0], self.q_weights)).float()
                for i in range(x_batch.shape[0])
            ]
        )
        return self.head(feats)


def train_hqnn(X_train, y_train, task="classification", hidden=CLASSICAL_DIM,
               n_steps=N_STEPS, lr=LR):
    model = HybridHQNN(hidden=hidden)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xtr = torch.tensor(X_train, dtype=torch.float32)
    ytr = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    loss_fn = nn.BCEWithLogitsLoss() if task == "classification" else nn.MSELoss()
    for _ in range(n_steps):
        opt.zero_grad()
        loss = loss_fn(model(Xtr), ytr)
        loss.backward()
        opt.step()
    return model


def eval_hqnn(model, X, y, task="classification"):
    Xe = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
        preds = model(Xe).numpy().ravel()
    if task == "classification":
        return accuracy_score(y, (preds > 0).astype(int))
    return r2_score(y, preds)


## 4. Dataset preview

In [ ]:
if qnlep_env_flag("QNLEP_REAL_DATA_FAST") and "QNLEP_REAL_DATA_NSAMPLES" not in os.environ:
    _ns_default = "100"
else:
    _ns_default = "160"
_ns = int(os.environ.get("QNLEP_REAL_DATA_NSAMPLES", _ns_default))

datasets = {
    'circles':     load_circles(n_samples=_ns),
    'moons':       load_moons(n_samples=_ns),
    'iris_1v2':    load_iris_binary(classes=(1, 2)),
    'mackey_glass': load_mackey_glass(
        n_train=int(os.environ.get("QNLEP_REAL_DATA_MG_TRAIN", "120")),
        n_test=int(os.environ.get("QNLEP_REAL_DATA_MG_TEST", "60")),
    ),
    'x_sin_x':     load_regression_1d(n_train=60, n_test=40),
}

task_map = {
    'circles': 'classification', 'moons': 'classification',
    'iris_1v2': 'classification', 'mackey_glass': 'regression',
    'x_sin_x': 'regression',
}

_ds_filter = os.environ.get("QNLEP_REAL_DATA_DATASETS", "").strip()
if qnlep_env_flag("QNLEP_REAL_DATA_FAST") and not _ds_filter:
    _keep = {"circles", "moons"}
elif _ds_filter:
    _keep = {x.strip() for x in _ds_filter.split(",") if x.strip()}
else:
    _keep = None
if _keep is not None:
    datasets = {k: v for k, v in datasets.items() if k in _keep}
    task_map = {k: task_map[k] for k in datasets}

_cls_preview = [n for n in ('circles', 'moons', 'iris_1v2') if n in datasets]
_n_prev = max(1, min(3, len(_cls_preview)))
fig, axes = plt.subplots(1, _n_prev, figsize=(4.6 * _n_prev, 3.5))
axes = np.atleast_1d(axes)

for ax, name in zip(axes, _cls_preview[:_n_prev]):
    Xtr, Xte, ytr, yte = datasets[name]
    X_all = np.vstack([Xtr, Xte])
    y_all = np.concatenate([ytr, yte])
    ax.scatter(X_all[:, 0], X_all[:, 1], c=y_all, cmap='bwr', s=15, alpha=0.7)
    ax.set_title(name)
    ax.set_xlabel('feature 0'); ax.set_ylabel('feature 1')

plt.suptitle('Classification datasets (scaled to [-π, π])', y=1.02)
plt.tight_layout()
plt.savefig('../figures/real_data_classification.pdf', bbox_inches='tight')
plt.show()

_reg_prev = [n for n in ('mackey_glass', 'x_sin_x') if n in datasets]
if _reg_prev:
    _nr = len(_reg_prev)
    fig, axes = plt.subplots(1, _nr, figsize=(5 * _nr, 3.5))
    axes = np.atleast_1d(axes)
    for ax, name in zip(axes, _reg_prev):
        Xtr, Xte, ytr, yte = datasets[name]
        ax.scatter(Xtr.ravel(), ytr, s=10, label='train', alpha=0.7)
        ax.scatter(Xte.ravel(), yte, s=10, label='test',  alpha=0.7, marker='x')
        ax.set_title(name)
        ax.set_xlabel('x(t)' if name == 'mackey_glass' else 'x')
        ax.set_ylabel('x(t+1)' if name == 'mackey_glass' else 'f(x)')
        ax.legend(fontsize=8)

    plt.suptitle('Regression datasets', y=1.02)
    plt.tight_layout()
    plt.savefig('../figures/real_data_regression.pdf', bbox_inches='tight')
    plt.show()

for name, (Xtr, Xte, ytr, yte) in datasets.items():
    print(f"{name:15s}  train={len(ytr):3d}  test={len(yte):3d}  "
          f"features={Xtr.shape[1]}  task={task_map[name]}")

## 5. Run all methods on all datasets

Note: circuits take a scalar `x` — for multi-feature datasets only the first
feature is used here. A multi-feature extension (encoding all features across
qubits) is in notebook 08 (Lorenz) and planned for the full QNLEP sweep.

In [ ]:
def _cap_train(X, y):
    cap = os.environ.get("QNLEP_REAL_DATA_MAX_TRAIN", "").strip()
    if not cap and qnlep_env_flag("QNLEP_REAL_DATA_FAST") and "QNLEP_REAL_DATA_MAX_TRAIN" not in os.environ:
        cap = "64"
    if not cap:
        return X, y
    n = int(cap)
    if len(y) <= n:
        return X, y
    rng = np.random.default_rng(RUN_SEED)
    idx = rng.choice(len(y), size=n, replace=False)
    return X[idx], y[idx]


all_results = {}   # all_results[dataset][method] = {'score': float, 'history': list, ...}

for ds_name, (Xtr, Xte, ytr, yte) in datasets.items():
    task = task_map[ds_name]
    all_results[ds_name] = {}
    Xtr_u, ytr_u = _cap_train(Xtr, ytr)
    print(f"\n=== {ds_name} ({task}) ===")

    # Classical MLP reference
    mlp = train_mlp(Xtr_u, ytr_u, task=task)
    mlp_score = eval_mlp(mlp, Xte, yte, task=task)
    all_results[ds_name]['classical_mlp'] = {'score': mlp_score, 'history': []}
    metric = 'acc' if task == 'classification' else 'R²'
    print(f"  classical_mlp          {metric}={mlp_score:.3f}")

    X1_tr = Xtr_u[:, :1]
    X1_te = Xte[:, :1]
    hqnn = train_hqnn(X1_tr, ytr_u, task=task)
    hq_score = eval_hqnn(hqnn, X1_te, yte, task=task)
    all_results[ds_name]['hybrid_hqnn'] = {'score': hq_score, 'history': []}
    print(f"  hybrid_hqnn            {metric}={hq_score:.3f}")

    # Quantum methods
    for method_name, (circuit, p_shape) in CIRCUITS.items():
        weights, history, steps = train_circuit(
            circuit, p_shape, Xtr_u, ytr_u, task=task
        )
        score = evaluate_circuit(circuit, weights, Xte, yte, task=task)
        n_params = int(np.prod(p_shape))
        # Param-efficiency TER: always >0; bonus when quality gate hits before budget
        _s = float(max(1, int(steps)))
        _np = float(max(1, int(n_params)))
        ter_raw = 1.0 / (_s * _np)
        if int(steps) < int(N_STEPS):
            ter_raw *= float(1.0 + 0.35 * (int(N_STEPS) - int(steps)) / max(float(N_STEPS), 1.0))

        all_results[ds_name][method_name] = {
            'score':   score,
            'history': history,
            'steps_to_thresh': steps,
            'n_params': n_params,
            'ter_raw':  ter_raw,
            'circuit_weights': np.asarray(weights, dtype=float),
        }
        print(f"  {method_name:25s}  {metric}={score:.3f}  steps={steps:3d}  n_params={n_params}")

## 6. Visualizations

### 6a. Score panels: classification (accuracy) vs regression ($R^2$)

Mackey--Glass and $x\sin x$ are **regression**; scores are **test $R^2$**, not accuracy. The combined PDF uses separate scales so Mackey--Glass is not forced into an accuracy-style $[0,1]$ colormap.

In [ ]:
method_names  = list(CIRCUITS.keys()) + ['classical_mlp', 'hybrid_hqnn']
dataset_names = list(datasets.keys())
cls_ds = [d for d in dataset_names if task_map[d] == 'classification']
reg_ds = [d for d in dataset_names if task_map[d] == 'regression']
reg_not_mg = [d for d in reg_ds if d != 'mackey_glass']
has_mg = 'mackey_glass' in reg_ds

n_panels = int(bool(cls_ds)) + int(bool(reg_not_mg)) + int(has_mg)
if n_panels == 0:
    raise RuntimeError('No datasets to plot')
# Panels: classification accuracy [0,1] | regression R^2 (x sin x) | Mackey--Glass R^2 bar
fig, axes = plt.subplots(1, n_panels, figsize=(4.2 * n_panels, len(method_names) * 0.72 + 1.3))
axes = np.atleast_1d(axes)
ip = 0

if cls_ds:
    M_cls = np.array([[all_results[ds][m]['score'] for ds in cls_ds] for m in method_names])
    im0 = axes[ip].imshow(M_cls, vmin=0.0, vmax=1.0, cmap='RdYlGn', aspect='auto')
    axes[ip].set_xticks(range(len(cls_ds)))
    axes[ip].set_xticklabels([d.replace('_', '\n') for d in cls_ds], fontsize=9)
    axes[ip].set_yticks(range(len(method_names)))
    axes[ip].set_yticklabels([m.replace('_', ' ') for m in method_names], fontsize=9)
    for i in range(len(method_names)):
        for j in range(len(cls_ds)):
            axes[ip].text(j, i, f"{M_cls[i, j]:.2f}", ha='center', va='center', fontsize=8)
    plt.colorbar(im0, ax=axes[ip], fraction=0.046, pad=0.04, label='Test accuracy')
    axes[ip].set_title('Classification', fontsize=10)
    ip += 1

if reg_not_mg:
    M_x = np.array([[all_results[ds][m]['score'] for ds in reg_not_mg] for m in method_names])
    xa, xb = float(M_x.min()), float(M_x.max())
    pad = 0.08 * (xb - xa + 1e-9)
    im1 = axes[ip].imshow(M_x, vmin=xa - pad, vmax=xb + pad, cmap='RdYlGn', aspect='auto')
    axes[ip].set_xticks(range(len(reg_not_mg)))
    axes[ip].set_xticklabels([d.replace('_', '\n') for d in reg_not_mg], fontsize=9)
    axes[ip].set_yticks(range(len(method_names)))
    axes[ip].set_yticklabels([])
    for i in range(len(method_names)):
        for j in range(len(reg_not_mg)):
            axes[ip].text(j, i, f"{M_x[i, j]:.2f}", ha='center', va='center', fontsize=8)
    plt.colorbar(im1, ax=axes[ip], fraction=0.046, pad=0.04, label=r'Test $R^2$')
    axes[ip].set_title(r'Regression ($x\sin x$)', fontsize=10)
    ip += 1

if has_mg:
    vals = np.array([all_results['mackey_glass'][m]['score'] for m in method_names], dtype=float)
    ypos = np.arange(len(method_names))
    axes[ip].barh(ypos, vals, color='steelblue', edgecolor='k', linewidth=0.3)
    axes[ip].set_yticks(ypos)
    axes[ip].set_yticklabels([m.replace('_', ' ') for m in method_names], fontsize=9)
    axes[ip].set_xlabel(r'Mackey--Glass test $R^2$ (not accuracy)', fontsize=9)
    axes[ip].set_title('Mackey--Glass (hard regression)', fontsize=10)
    axes[ip].axvline(0, color='k', lw=0.8, ls='--')
    axes[ip].grid(True, axis='x', alpha=0.3)

fig.suptitle('Real-data scores: accuracy vs.\ $R^2$ by task type', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('../figures/real_data_heatmap.pdf', bbox_inches='tight')
plt.show()

### 6b. Training curves per dataset (all methods on one plot)

In [ ]:
fig, axes = plt.subplots(1, len(datasets), figsize=(4 * len(datasets), 3.5))
axes = np.atleast_1d(axes)
colors = plt.cm.tab10.colors

for ax, ds_name in zip(axes, dataset_names):
    for i, m in enumerate(list(CIRCUITS.keys())):
        h = all_results[ds_name][m]['history']
        ax.semilogy(h, color=colors[i], lw=1.5, label=m.replace('_', ' '))
    ax.set_title(ds_name.replace('_', ' '), fontsize=9)
    ax.set_xlabel('Step')
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Loss (log scale)')
axes[-1].legend(fontsize=7, bbox_to_anchor=(1.05, 1))
fig.suptitle('Training curves — all quantum methods', y=1.02)
plt.tight_layout()
plt.savefig('../figures/real_data_training_curves.pdf', bbox_inches='tight')
plt.show()

### 6c. Decision boundary visualization (circles + moons)

Shows qualitatively how each method's learned function partitions the input space.

In [ ]:
def plot_decision_boundary(ax, circuit, weights, X_train, y_train, title):
    """Plot 2D decision boundary by evaluating circuit on a grid."""
    xx = np.linspace(-np.pi, np.pi, 40)
    preds_grid = np.array([float(circuit(weights, float(xi))) for xi in xx])

    # Show 1D prediction curve and threshold (crude 2D proxy using feature 0)
    ax.plot(xx, preds_grid, 'k-', lw=1.5, label='circuit output')
    ax.axhline(0, color='orange', ls='--', lw=1, label='threshold')
    ax.scatter(X_train[:, 0], [y_train[i] * 2 - 1 for i in range(len(y_train))],
               c=y_train, cmap='bwr', s=15, alpha=0.6, zorder=3)
    ax.set_title(title, fontsize=8)
    ax.set_ylim(-1.3, 1.3)
    ax.grid(True, alpha=0.3)


_bd_steps = int(os.environ.get("QNLEP_REAL_DATA_BOUNDARY_STEPS", "80"))
if qnlep_env_flag("QNLEP_REAL_DATA_FAST") and "QNLEP_REAL_DATA_BOUNDARY_STEPS" not in os.environ:
    _bd_steps = min(_bd_steps, 24)

for ds_name in [d for d in ('circles', 'moons') if d in datasets]:
    Xtr, Xte, ytr, yte = datasets[ds_name]
    Xtr_b, ytr_b = _cap_train(Xtr, ytr)
    n_methods = len(CIRCUITS)
    fig, axes = plt.subplots(1, n_methods, figsize=(3.5 * n_methods, 3.5))
    axes = np.atleast_1d(axes)

    for ax, (method_name, (circuit, p_shape)) in zip(axes, CIRCUITS.items()):
        rec = all_results[ds_name].get(method_name, {})
        w_stored = rec.get('circuit_weights')
        if w_stored is not None:
            weights_m = np.asarray(w_stored, dtype=float)
        else:
            weights_m = pnp.array(np.random.normal(0, 0.1, p_shape), requires_grad=True)
            weights_m, _, _ = train_circuit(circuit, p_shape, Xtr_b, ytr_b, task='classification',
                                            n_steps=_bd_steps)
        score = evaluate_circuit(circuit, weights_m, Xte, yte, task='classification')
        plot_decision_boundary(ax, circuit, weights_m, Xtr, ytr,
                               f"{method_name.replace('_',' ')}\nacc={score:.2f}")

    fig.suptitle(f'{ds_name} — circuit output vs feature 0', y=1.02)
    plt.tight_layout()
    plt.savefig(f'../figures/real_boundary_{ds_name}.pdf', bbox_inches='tight')
    plt.show()

### 6d. Mackey-Glass one-step prediction (QRC is expected to shine here)

In [ ]:
if 'mackey_glass' in datasets:
    Xtr_mg, Xte_mg, ytr_mg, yte_mg = datasets['mackey_glass']
    Xtr_mg_u, ytr_mg_u = _cap_train(Xtr_mg, ytr_mg)

    all_methods_mg = [('classical_mlp', None, None), ('hybrid_hqnn', None, None)] + [
        (name, circ, shape) for name, (circ, shape) in CIRCUITS.items()
    ]
    n_m = len(all_methods_mg)
    ncols = min(3, n_m)
    nrows = int(np.ceil(n_m / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3 * nrows), sharey=True)
    axes_flat = np.atleast_1d(axes).ravel()
    for ax in axes_flat[n_m:]:
        ax.set_visible(False)

    for ax, (method_name, circuit, p_shape) in zip(axes_flat, all_methods_mg):
        if method_name == 'classical_mlp':
            mlp  = train_mlp(Xtr_mg_u, ytr_mg_u, task='regression', hidden=16)
            with torch.no_grad():
                preds = mlp(torch.tensor(Xte_mg, dtype=torch.float32)).numpy().ravel()
            score = r2_score(yte_mg, preds)
        elif method_name == 'hybrid_hqnn':
            hq = train_hqnn(Xtr_mg_u[:, :1], ytr_mg_u, task='regression')
            with torch.no_grad():
                preds = hq(torch.tensor(Xte_mg[:, :1], dtype=torch.float32)).numpy().ravel()
            score = r2_score(yte_mg, preds)
        else:
            weights, _, _ = train_circuit(circuit, p_shape, Xtr_mg_u, ytr_mg_u, task='regression')
            preds  = np.array([float(circuit(weights, float(Xte_mg[i, 0])))
                               for i in range(len(Xte_mg))])
            score  = r2_score(yte_mg, preds)

        t = np.arange(len(yte_mg))
        ax.plot(t, yte_mg, 'k--', lw=1.5, label='true')
        ax.plot(t, preds,  'C0',  lw=2.0, label=f'pred R²={score:.2f}')
        ax.set_title(method_name.replace('_', ' '), fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    axes_flat[0].set_ylabel('x(t+1)  [scaled]')
    fig.suptitle('Mackey-Glass one-step prediction — all methods', y=1.02)
    plt.tight_layout()
    plt.savefig('../figures/mackey_glass_predictions.pdf', bbox_inches='tight')
    plt.show()
else:
    print("Skipping Mackey-Glass prediction panel (dataset not in current selection).")

## 7. TER comparison on real data

Trainability Efficiency Ratio on the hardest classification task (circles).

In [ ]:
focus_ds = (
    'circles' if 'circles' in all_results
    else 'moons' if 'moons' in all_results
    else next(iter(all_results))
)
methods_q = list(CIRCUITS.keys())
ter_vals  = []
for m in methods_q:
    r = all_results[focus_ds][m]
    ter_vals.append(r['ter_raw'])

max_ter = max(ter_vals) or 1.0
ter_norm = [t / max_ter for t in ter_vals]

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(
    [m.replace('_', '\n') for m in methods_q],
    ter_norm, color='steelblue', edgecolor='k'
)
ax.set_ylabel('TER (normalised, higher = more efficient)')
ax.set_title(f'Trainability Efficiency — {focus_ds}')
ax.set_ylim(0, 1.2)
for bar, t in zip(bars, ter_norm):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{t:.2f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('../figures/ter_real_data.pdf', bbox_inches='tight')
plt.show()

## 8. Export real-data scores (does not overwrite protocol `ter_scores.json`)

Lorenz TER for NLPS lives in `ter_scores.json` from notebook **08**. Here we write **`real_data_ter_scores.json`** (hardest available classification task among loaded datasets): per-method **param-efficiency** score `1/(steps·n_params)` with a mild bonus if the train quality gate (90% acc / 80% train R²) is reached before the step budget, then **max-normalised** across methods. **`real_data_scores.json`** holds per-dataset scores and step counts.

In [ ]:
# TER on hardest available classification task — separate file from protocol Lorenz TER (notebook 08)
focus_ds = (
    'circles' if 'circles' in all_results
    else 'moons' if 'moons' in all_results
    else next(iter(all_results))
)
ter_scores = {m: all_results[focus_ds][m].get('ter_raw', 0.0)
              for m in methods_q}
max_ter = max(ter_scores.values()) or 1.0
ter_scores_norm = {m: ter_scores[m] / max_ter for m in methods_q}

with open(RESULTS_DIR / 'real_data_ter_scores.json', 'w') as f:
    json.dump(ter_scores_norm, f, indent=2)

# Summary scores across all datasets
export = {
    ds: {m: {'score': all_results[ds][m]['score'],
              'steps': all_results[ds][m].get('steps_to_thresh', N_STEPS)}
         for m in all_results[ds]}
    for ds in all_results
}
with open(RESULTS_DIR / 'real_data_scores.json', 'w') as f:
    json.dump(export, f, indent=2)

print('Saved real_data_ter_scores.json and real_data_scores.json')
print()
print('Summary table:')
print(f"{'Method':25s}  ", end='')
for ds in dataset_names: print(f"{ds[:8]:>9s}", end='')
print()
print('-' * (25 + 9 * len(dataset_names) + 2))
for m in ['classical_mlp'] + methods_q:
    print(f"{m:25s}  ", end='')
    for ds in dataset_names:
        sc = all_results[ds].get(m, {}).get('score', float('nan'))
        print(f"{sc:9.3f}", end='')
    print()